# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mennahamdy0/flyrank-machine-learning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
!pwd

/content


In [7]:
!find . -name "content_refresh_anonymized.csv"

In [8]:
!git clone https://github.com/mennahamdy0/flyrank-machine-learning.git

Cloning into 'flyrank-machine-learning'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 128 (delta 43), reused 98 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.85 MiB | 13.60 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [9]:
%cd flyrank-machine-learning

/content/flyrank-machine-learning


In [10]:
!find . -name "content_refresh_anonymized.csv"

./data/raw/content_refresh_anonymized.csv


In [11]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)
print(df.head())


Dataset Shape: (30000, 44)
             content_id          client_id  search_volume  competition  \
0  content_304f48230142  client_f369cb89fc           10.0         0.67   
1  content_a1fb4e703a9e  client_4e07408562           90.0         0.01   
2  content_9aa793d4d895  client_7f2253d7e2            0.0         0.00   
3  content_331d6c4de07b  client_19581e27de           10.0         0.00   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0         0.00   

  competition_level   cpc     content_type    main_intent  word_count  \
0              HIGH  2.05  keyword article  transactional      3221.0   
1               LOW  0.05  keyword article  informational      2481.0   
2               LOW  0.00  keyword article  informational      3515.0   
3               LOW  0.00  keyword article     commercial         NaN   
4               LOW  0.00  keyword article  informational      2803.0   

   char_count  ... char_count_tier   ctr  avg_position  engagement_rate  \
0     20457.0 

### My Lane as an ML Task

I chose **Refresh / Content Opportunity Scoring**.

This is a **Scoring (Ranking)** task because the goal is to prioritize pages based on their need for content refresh. Instead of making a simple yes/no decision, the model should assign a score so the content team can review the highest-priority pages first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target or Proxy

The model will predict a **Refresh Priority Score** for each page.

Since there is no direct label showing whether a page needs refreshing, the score will be estimated using available search performance signals such as impressions, CTR, average position, trend direction, and content age.

This score will help rank pages from the highest to the lowest refresh priority.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df[[
    "content_age_days",
    "impressions_90d",
    "avg_position",
    "ctr",
    "trend_direction"
]].head()



,content_age_days,impressions_90d,avg_position,ctr,trend_direction
0,187,3803,10.6,0.76,down
1,445,15320,20.3,0.05,down
2,141,12581,36.5,0.09,down
3,463,11751,6.2,0.49,stable
4,263,19140,44.0,0.13,down


In [16]:
# Check the columns that may be useful for the target

target_features = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update"
]

print(df[target_features].describe())

       impressions_90d  avg_position           ctr  content_age_days  \
count     30000.000000   30000.00000  30000.000000       30000.00000   
mean       5200.366300      16.34238      0.510733         256.16780   
std       16838.019547      15.21679      3.279162         132.70793   
min           1.000000       0.00000      0.000000          90.00000   
25%          81.000000       6.20000      0.000000         132.00000   
50%         731.000000      10.80000      0.070000         236.00000   
75%        3615.250000      22.30000      0.290000         333.00000   
max      517715.000000     245.00000    100.000000         564.00000   

       days_since_last_update  
count            30000.000000  
mean                46.098300  
std                 42.078709  
min                  1.000000  
25%                 20.000000  
50%                 20.000000  
75%                104.000000  
max                373.000000  


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The success metric is Precision@50.

A good model should rank the most important pages near the top of the recommendation list, allowing content teams to review the highest-value pages first.

In [18]:
print("Rows in dataset:", len(df))
print("Average CTR:", round(df["ctr"].mean(),3))
print("Average Position:", round(df["avg_position"].mean(),2))

Rows in dataset: 30000
Average CTR: 0.511
Average Position: 16.34


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row represents one content page.

Each row contains search performance features that can help estimate whether the page should be refreshed.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df[[
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]].head()


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr
0,187,20,3803,10.6,0.76
1,445,25,15320,20.3,0.05
2,141,20,12581,36.5,0.09
3,463,22,11751,6.2,0.49
4,263,14,19140,44.0,0.13


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule cannot capture the complex relationship between multiple search signals.

Machine learning can combine several features together and learn better ranking patterns than simple if-else rules.

The output supports content teams in deciding which pages should be reviewed first.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Correlation between numerical features

corr = df[[
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update"
]].corr()

corr


,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update
impressions_90d,1.000000,-0.070786,-0.01895,-0.000743,0.081597
avg_position,-0.070786,1.000000,-0.07259,0.158238,0.070140
ctr,-0.018950,-0.072590,1.00000,0.009460,-0.020760
content_age_days,-0.000743,0.158238,0.00946,1.000000,0.038343
days_since_last_update,0.081597,0.070140,-0.02076,0.038343,1.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.